# Explainability and Clinical Alerts

In this notebook, we interpret the best performing model (e.g., LightGBM for the 24h prediction window) using SHAP (SHapley Additive exPlanations). 

Our explainability strategy aims to provide:
1. **Global insights:** Understanding which features drive the model's predictions overall (e.g., feature importance, SHAP summary plots).
2. **Local explanations:** Providing patient-specific insights to understand why a particular patient is flagged at high risk.
3. **Clinical Alerts:** Generating structured JSON alerts that highlight the top risk factors, which can be directly integrated into an EHR (Electronic Health Record) system to guide clinical decision-making.

In [ ]:
import os
import sys
import json
import pandas as pd

sys.path.append('..')
from src import models, features
from src.explain import run_explainability

# Interpret the Stage-2 LightGBM model at the 24h horizon on the test split.
test_df = pd.read_parquet('../data/processed/test.parquet')
stage1, stage2 = models.load_models('../models')
s2_feats = features.get_all_features(test_df)

H = 24
model = stage2[H]['lgbm']['model']
X_test = test_df[s2_feats]
y_true = test_df[f'aki_label_{H}h'].values
y_prob = model.predict_proba(X_test)[:, 1]

output_dir = '../outputs/explain_stage2_24h'
os.makedirs(output_dir, exist_ok=True)
print(f"Explaining Stage-2 LGBM @ {H}h on {len(X_test)} test rows, {len(s2_feats)} features.")

In [ ]:
print("Running explainability pipeline (SHAP summary + waterfalls + clinical alerts)...")

summary = run_explainability(
    model=model,
    X_test=X_test,
    y_true=y_true,
    y_prob=y_prob,
    feature_names=s2_feats,
    output_dir=output_dir,
)

print("\nTop global features:", ", ".join(summary['top_features'][:5]))
print("Sample alerts generated:", summary['alert_count'])
print("Artifacts saved to:", summary['output_dir'])

## Sample Clinical Alert

To show what the EHR would display, we can print a sample alert from the JSON file generated above. The alert includes a risk score and a breakdown of the specific factors driving that risk, providing actionable context to the clinician.

In [ ]:
# Load and print a sample alert to simulate the EHR display.
alerts_path = os.path.join(output_dir, 'sample_alerts.json')
try:
    with open(alerts_path, 'r') as f:
        alerts = json.load(f)

    if alerts:
        a = alerts[0]
        print("=" * 56)
        print("🚨 EHR CLINICAL ALERT — Sepsis-Associated AKI 🚨")
        print("=" * 56)
        print(f"Patient ID     : {a['patient_id']}")
        print(f"Risk level     : {a['risk_level']}  (score {a['risk_score']:.2f})")
        print(f"AKI prob (24h) : {a['aki_probability_24h']}")
        print("\nTop contributing factors:")
        for c in a['top_contributors']:
            print(f"  ⚠️  {c['feature']:32s} {c['direction']:8s} "
                  f"(value {c['value']}, SHAP {c['shap_impact']:+.3f})")
        print(f"\nRecommended action: {a['recommended_action']}")
        print("=" * 56)
    else:
        print("No alerts were generated.")

except FileNotFoundError:
    print("Alerts file not found. Ensure `run_explainability` ran successfully.")